This notebook demonstrates how to run the forest baseline User Defined Process

In [1]:
import logging

from utils import urls

import openeo

logging.basicConfig(level=logging.INFO)

In [2]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

INFO:openeo.config:Loaded openEO client config from sources: []


In [3]:
connection.authenticate_oidc()

INFO:openeo.rest.connection:Found OIDC providers: ['CDSE']
INFO:openeo.rest.connection:No OIDC provider given, but only one available: 'CDSE'. Using that one.
INFO:openeo.rest.connection:Using default client_id 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e' from OIDC provider 'CDSE' info.
INFO:openeo.rest.connection:Found refresh token: trying refresh token based authentication.
INFO:openeo.rest.auth.oidc:Doing 'refresh_token' token request 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token' with post data fields ['grant_type', 'client_id', 'refresh_token'] (client_id 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e')
INFO:openeo.rest.connection:Obtained tokens: ['token_type', 'access_token', 'expires_in', 'id_token', 'refresh_token', 'scope']
INFO:openeo.rest.auth.config:Storing refresh token for issuer 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE' (client 'sh-b1c3a958-52d4-40fe-a333-153595d1c71e')


Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [4]:
spatial_extent = {
    "west": 32.637423,
    "south": 0.060209,
    "east": 33.131945,
    "north": 0.291267,
}

# minimum canopy cover to be considered forest
canopy_cover_threshold = 30

# minimum likelihood to be considered natural forest
natural_forest_threshold = 0.08

# minimum connected area to be considered forest (m^2)
min_connected_area = 10000

In [5]:
cube = connection.datacube_from_process(
    "forest_baseline",
    namespace=urls.FOREST_BASELINE_UDP,
    spatial_extent=spatial_extent,
    canopy_cover_threshold=canopy_cover_threshold,
    natural_forest_threshold=natural_forest_threshold,
    min_connected_area=min_connected_area,
)

In [6]:
job = cube.create_job(out_format="GTiff")
# job = cube.create_job(out_format="netCDF")

In [7]:
job.start_and_wait()

0:00:00 Job 'j-26112317162446ec99e059819fcf7265': send 'start'
0:00:03 Job 'j-26112317162446ec99e059819fcf7265': created (progress 0%)
0:00:10 Job 'j-26112317162446ec99e059819fcf7265': queued (progress 0%)
0:00:17 Job 'j-26112317162446ec99e059819fcf7265': queued (progress 0%)
0:00:25 Job 'j-26112317162446ec99e059819fcf7265': running (progress N/A)
0:00:35 Job 'j-26112317162446ec99e059819fcf7265': running (progress N/A)
0:00:47 Job 'j-26112317162446ec99e059819fcf7265': running (progress N/A)
0:01:03 Job 'j-26112317162446ec99e059819fcf7265': running (progress N/A)
0:01:22 Job 'j-26112317162446ec99e059819fcf7265': running (progress N/A)
0:01:46 Job 'j-26112317162446ec99e059819fcf7265': running (progress N/A)
0:02:16 Job 'j-26112317162446ec99e059819fcf7265': running (progress N/A)
0:02:53 Job 'j-26112317162446ec99e059819fcf7265': running (progress N/A)
0:03:40 Job 'j-26112317162446ec99e059819fcf7265': finished (progress 100%)


<BatchJob job_id='j-26112317162446ec99e059819fcf7265'>

In [8]:
results = job.get_results()

In [9]:
!mkdir -p output-udp/
!rm -r output-udp/

In [10]:
results.download_files("output-udp/")

INFO:openeo.rest.job:Downloading job result asset 'openEO.tif' from https://s3.waw3-1.openeo.v1.dataspace.copernicus.eu/openeo-data-prod-waw4-1/batch_jobs/j-26112317162446ec99e059819fcf7265/openEO.tif?X-Proxy-Head-As-Get=true&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=ecfb5d44f118428ba8cf0229534ebb11%2F20261123%2Fwaw4-1%2Fs3%2Faws4_request&X-Amz-Date=20261123T172006Z&X-Amz-Expires=86400&X-Amz-SignedHeaders=host&X-Amz-Security-Token=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlX2FybiI6ImFybjpvcGVuZW93czppYW06Ojpyb2xlL29wZW5lby1kYXRhLXByb2Qtd2F3NC0xLXdvcmtzcGFjZSIsImluaXRpYWxfaXNzdWVyIjoib3BlbmVvLnByb2Qud2F3My0xLm9wZW5lby1pbnQudjEuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUiLCJodHRwczovL2F3cy5hbWF6b24uY29tL3RhZ3MiOnsicHJpbmNpcGFsX3RhZ3MiOnsiam9iX2lkIjpbImotMjYxMTIzMTcxNjI0NDZlYzk5ZTA1OTgxOWZjZjcyNjUiXSwidXNlcl9pZCI6WyIxM2YyYTg1Mi00YzFjLTQ4N2EtOGExYS0xYzA1YjE2M2I4YTIiXX0sInRyYW5zaXRpdmVfdGFnX2tleXMiOlsidXNlcl9pZCIsImpvYl9pZCJdfSwiaXNzIjoic3RzLndhdzMtMS5vcGVuZW8udjEuZGF0YXNwYWNlLmNvcGVybmljdXMu

[PosixPath('output-udp/openEO.tif'), PosixPath('output-udp/job-results.json')]